# 01 · Limpieza y preparación de datos

**Proyecto:** predicción del éxito de startups (Machine Learning, máster)

**Dataset:** Crunchbase (Kaggle `amirataha/startups`), fichero `data/raw/companies.zip`.

**Objetivo de este notebook:** pasar del CSV en bruto a un dataset limpio en `data/processed/startups_clean.csv`, listo para la EDA y el modelado.

Pasos:
1. Carga y primer vistazo
2. Duplicados
3. Definición del problema (variable objetivo)
4. Sesgo en los valores nulos
5. Fugas de información (*data leakage*)
6. Columnas sin valor predictivo
7. Fechas y valores imposibles
8. Ingeniería de variables
9. Revisión final de nulos
10. Guardado

> ⚠️ Aquí **no se imputa nada que dependa de los datos** (medias, medianas…). Eso se hace dentro del `Pipeline` de sklearn después del *split*, para no filtrar información del test al train.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# Ruta al zip: funciona tanto si el notebook está en la raíz del repo como en notebooks/
RAIZ = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RUTA_RAW = r"C:\TRASTEO\prosperStartup.AI\data\raw\companies.csv"
RUTA_CLEAN = r"C:\TRASTEO\prosperStartup.AI\data\processed\startups_clean.csv"
RUTA_RAW

## 1. Carga y primer vistazo

pandas lee el `.zip` directamente y respeta los saltos de línea que hay dentro de las descripciones (por eso **no** hay que abrirlo con Excel).

In [ ]:
raw = pd.read_csv(RUTA_RAW, low_memory=False)
print(raw.shape)
raw.head(3)

In [ ]:
raw.info()

In [ ]:
print(raw["entity_type"].value_counts(dropna=False), "\n")
print(raw["status"].value_counts(dropna=False))

In [ ]:
# % de nulos por columna en el dataset completo
(raw.isna().mean() * 100).round(1).sort_values(ascending=False)

## 2. Duplicados

Comprobamos que no haya filas repetidas ni empresas con el mismo identificador.

In [ ]:
print("Filas duplicadas completas:", raw.duplicated().sum())
print("IDs duplicados:            ", raw["id"].duplicated().sum())
print("Permalinks duplicados:     ", raw["permalink"].duplicated().sum())
print("Nombres duplicados:        ", raw["name"].duplicated().sum())

In [ ]:
# Los nombres repetidos pueden ser empresas distintas con el mismo nombre: los miramos
raw[raw["name"].duplicated(keep=False)].sort_values("name")[["id", "name", "category_code", "country_code", "status"]].head(10)

## 3. Definición del problema

- Las empresas `operating` **todavía no tienen desenlace**: no sabemos si triunfarán o fracasarán, así que no pueden etiquetarse. Las excluimos.
- **Éxito (1):** `acquired` o `ipo`.
- **Fracaso (0):** `closed`.

In [ ]:
df = raw[raw["status"].isin(["acquired", "ipo", "closed"])].copy()
df["success"] = (df["status"] != "closed").astype(int)

print(df.shape)
print(df["status"].value_counts())
print("\nProporción de clases:")
print(df["success"].value_counts(normalize=True).round(3))

## 4. Sesgo en los valores nulos

Comparamos el % de nulos de cada columna **según la clase**. Si una clase tiene muchos más nulos que la otra, el modelo puede aprender que "no tener datos" = éxito/fracaso, lo cual es un artefacto de cómo se recogieron los datos y no un patrón real.

In [ ]:
nulos_por_clase = df.isna().groupby(df["success"]).mean().T.round(2)
nulos_por_clase.columns = ["nulos_fracaso (0)", "nulos_exito (1)"]
nulos_por_clase["diferencia"] = (nulos_por_clase["nulos_exito (1)"] - nulos_por_clase["nulos_fracaso (0)"]).round(2)
nulos_por_clase.sort_values("diferencia", ascending=False)

**Interpretación:** muchas empresas de éxito (sobre todo `acquired`) son fichas casi vacías: probablemente se crearon en Crunchbase solo porque otra empresa las compró. Por ejemplo, `funding_rounds` está vacío en ~71 % de los éxitos frente a ~19 % de los fracasos.

**Decisión:** nos quedamos solo con las empresas que tienen información de financiación (`funding_rounds` no nulo). Así comparamos empresas con un nivel de información similar.

In [ ]:
antes = len(df)
df = df[df["funding_rounds"].notna()].copy()
print(f"Filas: {antes} -> {len(df)}")
print(df["success"].value_counts())
print(df["success"].value_counts(normalize=True).round(3))

In [ ]:
# Volvemos a comparar: ahora los nulos deberían estar mucho más equilibrados
nulos_por_clase = df.isna().groupby(df["success"]).mean().T.round(2)
nulos_por_clase.columns = ["nulos_fracaso (0)", "nulos_exito (1)"]
nulos_por_clase["diferencia"] = (nulos_por_clase["nulos_exito (1)"] - nulos_por_clase["nulos_fracaso (0)"]).round(2)
nulos_por_clase.sort_values("diferencia", ascending=False)

## 5. Fugas de información (*data leakage*)

Variables que contienen (directa o indirectamente) la respuesta, o que se conocen **después** del desenlace:

| Columna | Motivo |
|---|---|
| `closed_at` | Solo existe si la empresa ha cerrado → delata la clase |
| `ROI` | Solo existe si hubo adquisición/IPO → delata la clase |
| `status` | Es la variable objetivo en texto |
| `updated_at` | Fecha de última edición de la ficha, suele ser posterior al desenlace |
| `last_funding_at`, `last_milestone_at` | Pueden ser posteriores al desenlace (p. ej. una empresa adquirida deja de levantar rondas) |

In [ ]:
# Comprobación de la fuga de closed_at
pd.crosstab(df["closed_at"].notna(), df["success"], rownames=["tiene closed_at"])

In [ ]:
cols_leakage = ["closed_at", "ROI", "status", "updated_at", "last_funding_at", "last_milestone_at"]
df = df.drop(columns=cols_leakage)
df.shape

## 6. Columnas sin valor predictivo

- **Identificadores:** `id`, `Unnamed: 0.1`, `entity_id`, `parent_id`, `name`, `normalized_name`, `permalink`
- **URLs / logos:** `domain`, `homepage_url`, `logo_url`, `logo_width`, `logo_height`
- **Metadatos de Crunchbase:** `entity_type` (todas son *Company*), `created_by`, `created_at`
- **Inversiones hechas por la propia empresa:** `first_investment_at`, `last_investment_at`, `investment_rounds`, `invested_companies` (casi todo nulo)
- **Localización con demasiado detalle o muchos nulos:** `state_code`, `city`, `region`, `lat`, `lng` (nos quedamos con `country_code`)

In [ ]:
cols_inutiles = [
    "id", "Unnamed: 0.1", "entity_id", "parent_id", "name", "normalized_name", "permalink",
    "domain", "homepage_url", "logo_url", "logo_width", "logo_height",
    "entity_type", "created_by", "created_at",
    "first_investment_at", "last_investment_at", "investment_rounds", "invested_companies",
    "state_code", "city", "region", "lat", "lng",
]
print("% nulos de lo que eliminamos:")
print((df[cols_inutiles].isna().mean() * 100).round(1).sort_values(ascending=False))
df = df.drop(columns=cols_inutiles)
df.columns.tolist()

## 7. Fechas y valores imposibles

In [ ]:
for col in ["founded_at", "first_funding_at", "first_milestone_at"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df[["founded_at", "first_funding_at", "first_milestone_at"]].describe()

In [ ]:
# Edad de la empresa (en años) cuando recibe su primera ronda
df["edad_primera_ronda"] = (df["first_funding_at"] - df["founded_at"]).dt.days / 365.25
df["edad_primera_ronda"].describe()

In [ ]:
# Casos raros: primera ronda ANTES de fundarse, o empresas muy antiguas
print("Primera ronda antes de la fundación:", (df["edad_primera_ronda"] < 0).sum())
print("Fundadas antes de 1980:             ", (df["founded_at"].dt.year < 1980).sum())
print("Primera ronda antes de 1980:        ", (df["first_funding_at"].dt.year < 1980).sum())

df["edad_primera_ronda"].plot.hist(bins=60, figsize=(8, 3), title="Edad en la primera ronda (años)")
plt.show()

**Decisiones (podéis cambiarlas, pero justificadlas en la memoria):**
- Edad en la primera ronda **negativa** → dato incoherente → lo ponemos a nulo (se imputará después en el pipeline).
- Empresas fundadas antes de 1980 → no son "startups" en el sentido del proyecto → las eliminamos.

In [ ]:
df.loc[df["edad_primera_ronda"] < 0, "edad_primera_ronda"] = np.nan

antes = len(df)
df = df[~(df["founded_at"].dt.year < 1980)]   # los nulos de founded_at se mantienen
print(f"Filas: {antes} -> {len(df)}")

## 8. Ingeniería de variables

In [ ]:
# --- Fechas -> números ---
df["anio_fundacion"] = df["founded_at"].dt.year
df["anio_primera_ronda"] = df["first_funding_at"].dt.year
df["tiene_milestone"] = df["first_milestone_at"].notna().astype(int)

# --- Financiación ---
df["log_funding_total"] = np.log1p(df["funding_total_usd"])
df["log_funding_por_ronda"] = np.log1p(df["funding_total_usd"] / df["funding_rounds"])

# --- Presencia online / información pública ---
df["tiene_twitter"] = df["twitter_username"].notna().astype(int)
df["tiene_descripcion"] = df["description"].notna().astype(int)
df["n_tags"] = df["tag_list"].fillna("").apply(lambda s: len([t for t in s.split(",") if t.strip()]))

df[["anio_fundacion", "anio_primera_ronda", "log_funding_total", "log_funding_por_ronda", "n_tags"]].describe()

In [ ]:
# --- milestones y relationships ---
# En Crunchbase un nulo aquí significa "no tiene ninguno registrado", así que lo ponemos a 0.
# Es una regla fija (no se calcula con los datos), así que no provoca fuga aunque se haga antes del split.
df["milestones"] = df["milestones"].fillna(0)
df["relationships"] = df["relationships"].fillna(0)

In [ ]:
# --- Categóricas con muchas clases: agrupamos las poco frecuentes ---
print(df["category_code"].value_counts().head(25))

In [ ]:
MIN_CATEGORIA = 100   # categorías con menos empresas -> "other"
frec = df["category_code"].value_counts()
df["categoria"] = df["category_code"].where(df["category_code"].isin(frec[frec >= MIN_CATEGORIA].index), "other")
df["categoria"] = df["categoria"].fillna("unknown")

PAISES = ["USA", "GBR", "CAN"]
df["pais"] = np.where(df["country_code"].isin(PAISES), df["country_code"], "otros")
df.loc[df["country_code"].isna(), "pais"] = "unknown"

print(df["categoria"].value_counts(), "\n")
print(df["pais"].value_counts())

In [ ]:
# Tasa de éxito por categoría y país (primer vistazo, la EDA completa va en el siguiente notebook)
display(df.groupby("categoria")["success"].agg(["mean", "size"]).sort_values("mean"))
display(df.groupby("pais")["success"].agg(["mean", "size"]).sort_values("mean"))

In [ ]:
# Eliminamos las columnas originales que ya hemos transformado
cols_originales = [
    "founded_at", "first_funding_at", "first_milestone_at",
    "funding_total_usd", "twitter_username",
    "short_description", "description", "overview", "tag_list",
    "category_code", "country_code",
]
df = df.drop(columns=cols_originales)
df.head()

## 9. Revisión final de nulos

Los nulos que queden **no se imputan aquí**: se hará con `SimpleImputer` dentro del `Pipeline`, ajustado solo con el conjunto de entrenamiento.

In [ ]:
df.info()
print()
print((df.isna().mean() * 100).round(1).sort_values(ascending=False))

In [ ]:
print("Filas finales:", len(df))
print(df["success"].value_counts(normalize=True).round(3))

## 10. Guardado

In [ ]:
RUTA_CLEAN.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(RUTA_CLEAN, index=False)
print("Guardado en", RUTA_CLEAN, df.shape)

## Resumen para la memoria

1. Partimos de **196.553 empresas** de Crunchbase.
2. Eliminamos las `operating` (sin desenlace conocido) → **13.112** empresas con éxito/fracaso.
3. Detectamos un **sesgo en los nulos**: las empresas de éxito tenían muchos más campos vacíos. Filtramos por empresas con datos de financiación → nulos equilibrados entre clases.
4. Eliminamos variables con **fuga de información** (`closed_at`, `ROI`, fechas posteriores al desenlace).
5. Eliminamos identificadores, URLs, metadatos y columnas casi vacías.
6. Tratamos **valores imposibles** (rondas antes de la fundación, empresas anteriores a 1980).
7. Creamos variables: edad en la primera ronda, log de la financiación, financiación por ronda, presencia online, nº de tags, categoría y país agrupados.
8. Los nulos restantes se imputarán en el `Pipeline` tras el *split* train/test.